# rank0-only-side-effects — ex2: rank-0 download + barrier so every rank reads safely

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rank0-only-side-effects`. Running the final beacon cell reports progress against the `Distributed: rank-0-only side effects` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank-0-only side effects` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank0-only-side-effects`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank0-only-side-effects"
DD_SUBTOPIC = "Distributed: rank-0-only side effects"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Rank-0 side effects with `dist.barrier()` ordering — quick refresher

Pure `if rank == 0:` is enough for *fire-and-forget* writes (logs, checkpoints other ranks never read). When rank 0 produces something OTHER ranks must read (downloaded dataset, generated split file, tokenizer cache), you need a barrier so the readers don't race the writer:

```python
if rank == 0:
    download_dataset(url, target)     # rank-0-only side effect
dist.barrier()                        # everyone waits here
data = read_dataset(target)           # every rank can now read safely
```

**Why barrier on every rank.** `dist.barrier()` is a collective — every rank must call it, or the call deadlocks. Rank > 0 hits the barrier immediately and blocks; rank 0 does the download first, THEN hits the barrier, which unblocks everyone.

**Order matters at one place only.** The barrier goes BETWEEN the rank-0 side effect and the all-rank read. Pre-barrier, ranks > 0 wait. Post-barrier, the produced file is guaranteed to exist for every rank.

### Exercise 2 — rank-0 download + barrier so every rank reads safely

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the writer-readers ordering pattern by combining the `if rank == 0:` guard for a download with a `dist.barrier()` so rank > 0's read is guaranteed to happen AFTER rank 0's write.
> Keywords: rank-0, barrier, ordering, download, shared-resource
> ```

**KCs targeted:** `rank0-side-effect-with-barrier`, `barrier-orders-writer-before-readers`

Implement `ex2_rank0_download_then_all_read(rank, world_size, dist_module, downloader, reader, log)`. The producer-consumer pattern that EVERY distributed training script needs once at startup:

1. **Rank 0 only — download.** If `rank == 0`, call `downloader()` and then `log('rank0-downloaded')` so the test can verify the order.
2. **Every rank — barrier.** Call `dist_module.barrier()` unconditionally. Ranks > 0 hit this first and block; rank 0 hits it after step 1 and unblocks everyone.
3. **Every rank — read.** Call `value = reader()` (the file is now guaranteed to exist on shared storage), then `log(f'rank{rank}-read-{value}')`. Return `value`.

Critical ordering invariants the test checks:
- `rank0-downloaded` appears in the log BEFORE any `rank*-read-...` entry.
- `downloader()` is called EXACTLY ONCE across all ranks (only rank 0 invokes it).
- `reader()` is called once per rank (every rank reads).

Input: `rank`, `world_size` ints; `dist_module` (mocked dist); `downloader`, `reader`, `log` callables.
Output: the value returned by `reader()` — same on every rank (reader returns a constant in the mock).

In [ ]:
def ex2_rank0_download_then_all_read(rank, world_size, dist_module,
                                    downloader, reader, log):
    """Rank-0 download, barrier, then every rank reads."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import types as _types
    import torch as _t_for_fake


    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'


    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            self.scratch = {}
            self.tls = threading.local()
            self.results = [None] * world_size
            # rank-0-only side-effect log (per-call log lines for tests to inspect)
            self.side_effects = []

        def _reduce_op(self, bag, op):
            if op == 'SUM':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = out + x
                return out
            if op == 'MAX':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = _t_for_fake.maximum(out, x)
                return out
            if op == 'MIN':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = _t_for_fake.minimum(out, x)
                return out
            if op == 'PROD':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = out * x
                return out
            raise ValueError(f'unknown fake op {op!r}')

        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['ar']
            reduced = self._reduce_op(bag, op)
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()

        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['rd']
                tensor.copy_(self._reduce_op(bag, op))
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()

        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()

        def gather(self, tensor, gather_list, dst):
            """Mock dist.gather — only dst's gather_list is populated."""
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('gth', [None] * self.world_size)
                self.scratch['gth'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['gth']
                for i, src_tensor in enumerate(bag):
                    gather_list[i].copy_(src_tensor)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('gth', None)
            self.barrier.wait()

        def all_gather(self, gather_list, tensor):
            """Mock dist.all_gather — every rank's gather_list is populated."""
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('agth', [None] * self.world_size)
                self.scratch['agth'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['agth']
            for i, src_tensor in enumerate(bag):
                gather_list[i].copy_(src_tensor)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('agth', None)
            self.barrier.wait()

        def barrier_op(self):
            self.barrier.wait()


    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size

        def _runner(rank):
            world.tls.rank = rank
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.gather = lambda tensor, gather_list, dst: world.gather(tensor, gather_list, dst)
            fake_dist.all_gather = lambda gather_list, tensor: world.all_gather(gather_list, tensor)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.init_process_group = lambda **kw: None
            fake_dist.destroy_process_group = lambda: None
            try:
                worker_fn(rank, world_size, fake_dist, world, *extra_args)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())

        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world.results


    # Shared mutable state — guarded by the world.lock that the harness exposes.
    _log_lines = []
    _log_lock = threading.Lock()
    _download_calls = [0]
    _read_calls = [0]

    def _downloader():
        with _log_lock:
            _download_calls[0] += 1
        return None

    def _reader():
        with _log_lock:
            _read_calls[0] += 1
        return 'payload-42'

    def _log(msg):
        with _log_lock:
            _log_lines.append(msg)

    def _worker(rank, world_size, dist_module, world):
        val = ex2_rank0_download_then_all_read(rank, world_size, dist_module,
                                              _downloader, _reader, _log)
        world.results[rank] = val

    results = _run_fake_world(_worker, 4)

    # Every rank got the same payload.
    for rank, r in enumerate(results):
        assert r == 'payload-42', f'rank {rank}: got {r!r}, expected payload-42'

    # downloader called exactly once.
    assert _download_calls[0] == 1, (
        f'downloader fired {_download_calls[0]} times — must be 1 (rank 0 only)'
    )
    # reader called once per rank.
    assert _read_calls[0] == 4, f'reader fired {_read_calls[0]} times — expected 4'

    # Ordering invariant: rank0-downloaded must appear BEFORE any rank*-read entry.
    download_idx = _log_lines.index('rank0-downloaded')
    read_idxs = [i for i, m in enumerate(_log_lines) if m.startswith('rank') and '-read-' in m]
    assert len(read_idxs) == 4, f'expected 4 read log entries, got {len(read_idxs)}'
    assert download_idx < min(read_idxs), (
        f'rank0-downloaded must precede every read; got download@{download_idx}, '
        f'min read@{min(read_idxs)}.  Did you forget the barrier?'
    )

    # Reset + re-run with world_size=1 — rank 0 IS everyone; barrier still works.
    _log_lines.clear(); _download_calls[0] = 0; _read_calls[0] = 0
    results1 = _run_fake_world(_worker, 1)
    assert results1[0] == 'payload-42'
    assert _download_calls[0] == 1 and _read_calls[0] == 1

    # Sanity: world_size=2 — only one downloader call, both ranks read.
    _log_lines.clear(); _download_calls[0] = 0; _read_calls[0] = 0
    results2 = _run_fake_world(_worker, 2)
    assert _download_calls[0] == 1, f'ws=2: downloader fired {_download_calls[0]}'
    assert _read_calls[0] == 2
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_rank0_download_then_all_read(rank, world_size, dist_module,
                                    downloader, reader, log):
    if rank == 0:
        downloader()
        log('rank0-downloaded')
    dist_module.barrier()
    value = reader()
    log(f'rank{rank}-read-{value}')
    return value
```

**Without the barrier, rank 1 races.** A bare `if rank == 0:` guarded write returns immediately on rank 0; meanwhile rank 1 tries to call `reader()` before rank 0 has finished writing. The file may be missing, half-written, or — worst — exist but with stale contents from a previous run. The barrier is the one line that prevents the race.

**Every rank must call barrier.** A barrier on rank 0 alone doesn't help — barrier is a collective. If rank 1 skips it, rank 0 hangs forever waiting for the world to catch up.

**Real DDP recipe:** rank 0 calls `download + write to NFS / S3`, `dist.barrier()`, every rank reads from the shared store. Same shape as this drill — just bigger downloader.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()